# Fine-Tuned DistilBERT Classifier Demo

This notebook demonstrates how to load and run predictions with the fine-tuned customer support text classifier model.

## 1. Load the Model & Tokenizer
We can load the model either from the local `model_output` folder or directly from the Hugging Face Hub at `oberbics/test_bert`.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Choose either the local path or the Hugging Face hub path
model_path = "model_output"  # or "oberbics/test_bert"

print(f"Loading model and tokenizer from '{model_path}'...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

# Ensure CPU usage
device = torch.device("cpu")
model.to(device)
model.eval()
print("Model loaded successfully!")

## 2. Define the Inference Function
Let's create a helper function that tokenizes the input, performs classification, and returns the predicted label and confidence score.

In [ ]:
def predict(text):
    # Tokenize input text
    inputs = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=64,
        return_tensors="pt"
    )
    
    # Run prediction
    with torch.no_grad():
        inputs = {key: val.to(device) for key, val in inputs.items()}
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
        pred_id = torch.argmax(probs, dim=1).item()
        
    # Map ID to class label
    id2label = model.config.id2label
    predicted_label = id2label[pred_id]
    confidence = probs[0][pred_id].item()
    
    return predicted_label, confidence

## 3. Test Predictions
Now we can try running it on a few test sentences representing the 3 target classes:
- `billing`
- `technical_support`
- `account_management`

In [ ]:
test_sentences = [
    "I was charged twice for premium subscription",
    "The mobile app keeps crashing on login",
    "I want to delete my account permanently",
    "Can you reset my password please?",
    "Why is my bill so high this month?"
]

for sentence in test_sentences:
    label, confidence = predict(sentence)
    print(f"Text: '{sentence}'")
    print(f"--> Predicted: {label} (Confidence: {confidence:.4f})\n")